# Single-drug FAERS: frozen ChemBERTa experiment

This compares a frozen pretrained molecular encoder plus a Ridge regression head against the training-mean baseline and five-seed Morgan ExtraTrees. It selects Ridge regularization using a scaffold-disjoint split of 2025 only. The 2026 H1 results are exploratory because they have already been inspected. The target is a FAERS serious-outcome report fraction, not clinical toxicity risk.

In [ ]:
from pathlib import Path
from datetime import datetime
import os, subprocess, sys

DATA_CANDIDATES = [Path("/content/gdrive_auditddi_toxicity/MyDrive/Projects/auditddi"), Path("/content/gdrive_auditddi/MyDrive/Projects/auditddi"), Path("/content/drive/MyDrive/Projects/auditddi")]
DATA = next((p for p in DATA_CANDIDATES if p.is_dir()), None)
if DATA is None:
    from google.colab import drive
    MOUNT = Path("/content/gdrive_auditddi_chemberta")
    if MOUNT.exists() and any(MOUNT.iterdir()): MOUNT = Path("/content/gdrive_auditddi_chemberta2")
    drive.mount(str(MOUNT))
    DATA = MOUNT / "MyDrive/Projects/auditddi"
assert DATA.is_dir(), f"Drive data root not found: {DATA}"

REPO = Path("/content/auditddi_toxicity_chemberta")
if not REPO.exists():
    subprocess.run(["git", "clone", "--branch", "main", "https://github.com/akhilau-git/auditddi.git", str(REPO)], check=True)
else:
    dirty = subprocess.run(["git", "-C", str(REPO), "status", "--porcelain"], capture_output=True, text=True, check=True).stdout.strip()
    assert not dirty, "Code checkout has local changes. Use a fresh runtime or clean checkout before pulling."
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "origin", "main"], check=True)
commit = subprocess.run(["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"], capture_output=True, text=True, check=True).stdout.strip()
print("Data root:", DATA)
print("Code commit:", commit)


In [ ]:
%pip install -q "transformers>=5.0,<6"
print("Transformers installation complete. Restart the runtime only if Colab prompts you to do so.")

In [ ]:
import py_compile
TRAINER = REPO / "scripts/train_single_drug_toxicity_chemberta.py"
py_compile.compile(str(TRAINER), doraise=True)
LABEL_ROOT = DATA / "results/pillars/single_drug_toxicity/label_builds"
label_dirs = sorted(p for p in LABEL_ROOT.glob("faers_2025_train_2026h1_test_*") if (p / "train_2025_suspect_roles.csv").is_file() and (p / "test_new_cases_2026h1_suspect_roles.csv").is_file())
assert label_dirs, f"Completed train/test labels not found under {LABEL_ROOT}"
LABEL_DIR = label_dirs[-1]
baseline_dirs = sorted(p for p in (DATA / "results/pillars/single_drug_toxicity/temporal_models").glob("*five_seed_*") if (p / "temporal_predictions.csv").is_file())
assert baseline_dirs, "Five-seed ExtraTrees predictions not found. Complete that run first."
BASELINE_PREDICTIONS = baseline_dirs[-1] / "temporal_predictions.csv"
OUT = DATA / "results/pillars/single_drug_toxicity/pretrained_models" / f"chemberta_frozen_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
OUT.mkdir(parents=True, exist_ok=False)
command = [sys.executable, "-u", str(TRAINER), "--train-labels", str(LABEL_DIR / "train_2025_suspect_roles.csv"), "--test-labels", str(LABEL_DIR / "test_new_cases_2026h1_suspect_roles.csv"), "--output-dir", str(OUT), "--benchmark-predictions", str(BASELINE_PREDICTIONS), "--device", "auto"]
log_path = OUT / "chemberta_training.log"
print("Using labels:", LABEL_DIR)
print("ExtraTrees comparison:", BASELINE_PREDICTIONS)
print("Output folder:", OUT)
print("Live log:", log_path)
with subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as process, log_path.open("w", encoding="utf-8") as log:
    for line in process.stdout:
        print(line, end="", flush=True)
        log.write(line); log.flush()
    return_code = process.wait()
if return_code != 0: raise RuntimeError(f"Frozen ChemBERTa run failed with code {return_code}. See {log_path}")
print("Metrics:", OUT / "pretrained_temporal_metrics.csv")
print("Manifest:", OUT / "pretrained_run_manifest.json")